# Step 2:构造因子

In [12]:
import numpy as np
import pandas as pd
from pathlib import Path
from statsmodels.tsa.stattools import adfuller
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')



---
## 1. import & data loading

In [13]:
DATA_PATH=Path('data/spy_minute_clean.parquet')
OUTPUT_DIR=Path('data')
OUTPUT_DIR.mkdir(exist_ok=True)
df=pd.read_parquet(DATA_PATH)
df.index=pd.to_datetime(df.index)
if df.index.tz is None:
    df.index = df.index.tz_localize('US/Eastern')
elif str(df.index.tz) != 'US/Eastern':
    df.index = df.index.tz_convert('US/Eastern')

df['date'] = df.index.date
df['minute'] = df.index.strftime('%H:%M')

---
## 2. 全局参数与 train 段定义

In [14]:
WINSOR_LOW, WINSOR_HIGH = -0.01, 0.01
BASELINE_WINDOW = 252
ZSCORE_WINDOW = 252

---
## 3. 因子构造函数库

In [15]:
def prepare_minute_returns(df):
    out = df.copy()
    out['logp'] = np.log(out['close'])
    out['r_raw'] = out.groupby('date')['logp'].diff()
    out['r_win'] = out['r_raw'].clip(lower=WINSOR_LOW, upper=WINSOR_HIGH)
    out['minute'] = out.index.strftime('%H:%M')
    return out


def apply_rolling_volume_deseasonalize(df_minute, window=BASELINE_WINDOW):
    out = df_minute.copy()
    pivot = out.pivot_table(index='date', columns='minute', values='volume', aggfunc='first')
    pivot = pivot.sort_index()
    rolling_baseline = pivot.shift(1).rolling(window=window, min_periods=window).mean()
    baseline_long = rolling_baseline.stack().rename('vol_baseline')
    baseline_long.index.names = ['date', 'minute']
    out = out.reset_index().merge(baseline_long.reset_index(), on=['date', 'minute'], how='left')
    out['v_tilde'] = out['volume'] / out['vol_baseline'].replace(0, np.nan)
    out = out.set_index(out.columns[0]).sort_index()
    out.index.name = df_minute.index.name
    return out, rolling_baseline


def _gini(arr):
    a = np.asarray(arr, dtype=float)
    a = a[~np.isnan(a)]
    if len(a) == 0 or np.all(a == 0):
        return np.nan
    a = np.sort(a)
    n = len(a)
    if a[0] < 0:
        a = a - a[0]
    cum = np.cumsum(a)
    if cum[-1] == 0:
        return np.nan
    return (n + 1 - 2 * np.sum(cum) / cum[-1]) / n


def compute_intraday_factors(df_minute):
    factors = {}
    intraday_mask = (df_minute['minute'] >= '09:31') & (df_minute['minute'] <= '15:59')
    open30_mask = (df_minute['minute'] >= '09:31') & (df_minute['minute'] <= '10:00')
    close30_mask = (df_minute['minute'] >= '15:30') & (df_minute['minute'] <= '15:59')
    close30_nc_mask = (df_minute['minute'] >= '15:30') & (df_minute['minute'] <= '15:58')

    sub_intra = df_minute.loc[intraday_mask]
    sub_open30 = df_minute.loc[open30_mask]
    sub_close30 = df_minute.loc[close30_mask]
    sub_close30_nc = df_minute.loc[close30_nc_mask]

    rv_open30 = sub_open30.groupby('date').apply(lambda x: (x['r_win'] ** 2).sum())
    rv_close30 = sub_close30.groupby('date').apply(lambda x: (x['r_win'] ** 2).sum())
    rv_close30_nc = sub_close30_nc.groupby('date').apply(lambda x: (x['r_win'] ** 2).sum())
    factors['RV_open30'] = rv_open30
    factors['RV_close30'] = rv_close30
    factors['RV_close30_nc'] = rv_close30_nc
    factors['VolPattern'] = rv_close30 / rv_open30.replace(0, np.nan)

    def _path_eff(x):
        r = x['r_win'].values
        denom = np.sum(np.abs(r))
        return np.abs(np.sum(r)) / denom if denom > 0 else np.nan

    def _vwap_cross(x):
        diff = x['close'].values - x['vwap'].values
        sgn = np.sign(diff)
        return int(np.sum(sgn[1:] * sgn[:-1] < 0))

    def _close_loc(x):
        hi, lo = x['high'].max(), x['low'].min()
        cl = x['close'].iloc[-1]
        return (cl - lo) / (hi - lo) if hi > lo else np.nan

    factors['PathEff'] = sub_intra.groupby('date').apply(_path_eff)
    factors['VWAP_Cross'] = sub_intra.groupby('date').apply(_vwap_cross)
    factors['CloseLoc'] = df_minute.groupby('date').apply(_close_loc)

    def _signed_volume(x, vol_col):
        return float(np.sum(x[vol_col].values * np.sign(x['r_win'].values)))

    def _vol_gini(x, vol_col):
        return _gini(x[vol_col].values)

    def _price_vol_corr(x, vol_col):
        ar = np.abs(x['r_win'].values)
        v = x[vol_col].values
        mask = ~(np.isnan(ar) | np.isnan(v))
        ar, v = ar[mask], v[mask]
        if len(ar) < 2 or np.std(ar) == 0 or np.std(v) == 0:
            return np.nan
        return float(np.corrcoef(ar, v)[0, 1])

    factors['SignedVolume'] = sub_intra.groupby('date').apply(lambda x: _signed_volume(x, 'v_tilde'))
    factors['VolGini'] = sub_intra.groupby('date').apply(lambda x: _vol_gini(x, 'v_tilde'))
    factors['PriceVolCorr'] = sub_intra.groupby('date').apply(lambda x: _price_vol_corr(x, 'v_tilde'))

    factors['SignedVolume_raw'] = sub_intra.groupby('date').apply(lambda x: _signed_volume(x, 'volume'))
    factors['VolGini_raw'] = sub_intra.groupby('date').apply(lambda x: _vol_gini(x, 'volume'))
    factors['PriceVolCorr_raw'] = sub_intra.groupby('date').apply(lambda x: _price_vol_corr(x, 'volume'))

    def _rv(x):
        return float((x['r_win'] ** 2).sum())

    def _bv(x):
        r = x['r_win'].values
        N = len(r)
        if N < 2:
            return np.nan
        absr = np.abs(r)
        return (np.pi / 2.0) * (N / (N - 1)) * float(np.sum(absr[1:] * absr[:-1]))

    def _signed_jump(x):
        r = x['r_win'].values
        rv_pos = float(np.sum(r[r > 0] ** 2))
        rv_neg = float(np.sum(r[r < 0] ** 2))
        return rv_pos - rv_neg

    rv_full = sub_intra.groupby('date').apply(_rv)
    bv_full = sub_intra.groupby('date').apply(_bv)
    factors['BV'] = bv_full
    factors['Jump'] = (rv_full - bv_full).clip(lower=0)
    factors['SignedJump'] = sub_intra.groupby('date').apply(_signed_jump)

    def _bv_raw(x):
        r = x['r_raw'].values
        N = len(r)
        if N < 2:
            return np.nan
        absr = np.abs(r)
        return (np.pi / 2.0) * (N / (N - 1)) * float(np.nansum(absr[1:] * absr[:-1]))

    def _rv_raw(x):
        return float(np.nansum(x['r_raw'].values ** 2))

    rv_raw_full = sub_intra.groupby('date').apply(_rv_raw)
    bv_raw_full = sub_intra.groupby('date').apply(_bv_raw)
    factors['Jump_raw'] = (rv_raw_full - bv_raw_full).clip(lower=0)

    def _vr_5_1(x):
        r1 = x['r_win'].values
        if len(r1) < 25:
            return np.nan
        n5 = (len(r1) // 5) * 5
        if n5 == 0:
            return np.nan
        r1c = r1[:n5]
        r5 = r1c.reshape(-1, 5).sum(axis=1)
        v1 = np.var(r1c, ddof=1)
        v5 = np.var(r5, ddof=1)
        if v1 == 0:
            return np.nan
        return float(v5 / (5.0 * v1))

    factors['VR_5_1'] = sub_intra.groupby('date').apply(_vr_5_1)

    daily = pd.DataFrame(factors)
    daily.index = pd.to_datetime(daily.index)
    daily.index.name = 'date'
    return daily


def compute_overnight(df_minute):
    open_bars = df_minute[df_minute['minute'] == '09:30'].copy()
    close_bars = df_minute[df_minute['minute'] == '15:59'].copy()
    open_bars = open_bars[['date', 'open']].rename(columns={'open': 'open_0930'})
    close_bars = close_bars[['date', 'close']].rename(columns={'close': 'close_1559'})
    open_bars = open_bars.set_index('date')
    close_bars = close_bars.set_index('date')
    merged = open_bars.join(close_bars, how='outer').sort_index()
    merged['prev_close'] = merged['close_1559'].shift(1)
    overnight = (merged['open_0930'] - merged['prev_close']) / merged['prev_close']
    overnight.index = pd.to_datetime(overnight.index)
    overnight.index.name = 'date'
    return overnight.rename('Overnight')


def add_zscore_versions(daily_factors, cols, window=ZSCORE_WINDOW):
    out = daily_factors.copy()
    for c in cols:
        s = out[c]
        mu = s.rolling(window, min_periods=window).mean()
        sd = s.rolling(window, min_periods=window).std()
        out[f'{c}_z252'] = (s - mu) / sd.replace(0, np.nan)
    return out

In [16]:
df_min = prepare_minute_returns(df)
df_min, rolling_baseline = apply_rolling_volume_deseasonalize(df_min, window=BASELINE_WINDOW)

print(f"Rolling baseline shape: {rolling_baseline.shape}")
print(f"Baseline first valid date: {rolling_baseline.dropna(how='all').index.min()}")


daily = compute_intraday_factors(df_min)
overnight = compute_overnight(df_min)
daily = daily.join(overnight, how='left')

c_main_cols = ['SignedVolume', 'VolGini', 'PriceVolCorr']
daily = add_zscore_versions(daily, c_main_cols, window=ZSCORE_WINDOW)

#加上对数收益率、动量因子
daily_close=(df_min[df_min['minute']=='15:59'].groupby('date')['logp'].last())
daily_close.index=pd.to_datetime(daily_close.index)
daily_close.index.name='date'
daily['Ret_1d']  = daily_close.diff(1)          # 上一日对数收益率
daily['Mom_20d'] = daily_close.diff(20)          # 20日累计对数收益率
daily['Mom_60d'] = daily_close.diff(60)          # 60日累计对数收益率

print(f"\nDaily factors shape: {daily.shape}")
print(f"Columns ({len(daily.columns)}): {list(daily.columns)}")
print(f"\nFirst valid index per column:")
print(daily.apply(lambda s: s.first_valid_index()))

Rolling baseline shape: (2745, 390)
Baseline first valid date: 2016-01-06

Daily factors shape: (2745, 25)
Columns (25): ['RV_open30', 'RV_close30', 'RV_close30_nc', 'VolPattern', 'PathEff', 'VWAP_Cross', 'CloseLoc', 'SignedVolume', 'VolGini', 'PriceVolCorr', 'SignedVolume_raw', 'VolGini_raw', 'PriceVolCorr_raw', 'BV', 'Jump', 'SignedJump', 'Jump_raw', 'VR_5_1', 'Overnight', 'SignedVolume_z252', 'VolGini_z252', 'PriceVolCorr_z252', 'Ret_1d', 'Mom_20d', 'Mom_60d']

First valid index per column:
RV_open30           2015-01-02
RV_close30          2015-01-02
RV_close30_nc       2015-01-02
VolPattern          2015-01-02
PathEff             2015-01-02
VWAP_Cross          2015-01-02
CloseLoc            2015-01-02
SignedVolume        2016-01-06
VolGini             2016-01-06
PriceVolCorr        2016-01-06
SignedVolume_raw    2015-01-02
VolGini_raw         2015-01-02
PriceVolCorr_raw    2015-01-02
BV                  2015-01-02
Jump                2015-01-02
SignedJump          2015-01-02
Jump_

---
## 5.描述性统计

In [17]:
desc = daily.describe().T
desc['skew'] = daily.skew()
desc['kurtosis'] = daily.kurtosis()
desc['n_nan'] = daily.isna().sum()
desc[['count', 'mean', 'std', 'min', 'max', 'skew', 'kurtosis', 'n_nan']].round(4)

,count,mean,std,min,max,skew,kurtosis,n_nan
RV_open30,2745.0,0.0000,0.000000e+00,0.000000e+00,6.000000e-04,12.9551,236.5056,0
RV_close30,2745.0,0.0000,0.000000e+00,0.000000e+00,7.000000e-04,12.1351,223.3628,0
RV_close30_nc,2745.0,0.0000,0.000000e+00,0.000000e+00,6.000000e-04,11.3988,196.9876,0
VolPattern,2745.0,1.1116,3.366100e+00,6.600000e-03,1.008879e+02,17.6152,408.2747,0
PathEff,2745.0,0.0549,4.160000e-02,0.000000e+00,2.596000e-01,0.9116,0.4933,0
VWAP_Cross,2745.0,194.4033,1.100020e+01,1.530000e+02,2.410000e+02,-0.1156,0.0827,0
CloseLoc,2745.0,0.5999,3.176000e-01,0.000000e+00,1.000000e+00,-0.4111,-1.2137,0
SignedVolume,2493.0,1.2953,3.421260e+01,-2.323045e+02,3.104046e+02,-0.2986,6.3654,252
VolGini,2493.0,0.3591,7.420000e-02,1.973000e-01,7.747000e-01,0.9175,1.8214,252
PriceVolCorr,2493.0,0.2435,1.276000e-01,-3.850000e-02,7.809000e-01,0.7028,0.7609,252


In [ ]:
factor_meta = {
    'RV_open30':         {'category': 'A', 'version': 'main', 'burn_in': 0},
    'RV_close30':        {'category': 'A', 'version': 'main', 'burn_in': 0},
    'RV_close30_nc':     {'category': 'A', 'version': 'nc',   'burn_in': 0},
    'VolPattern':        {'category': 'A', 'version': 'main', 'burn_in': 0},
    'PathEff':           {'category': 'B', 'version': 'main', 'burn_in': 0},
    'VWAP_Cross':        {'category': 'B', 'version': 'main', 'burn_in': 0},
    'CloseLoc':          {'category': 'B', 'version': 'main', 'burn_in': 0},
    'SignedVolume':      {'category': 'C', 'version': 'main', 'burn_in': 252},
    'VolGini':           {'category': 'C', 'version': 'main', 'burn_in': 252},
    'PriceVolCorr':      {'category': 'C', 'version': 'main', 'burn_in': 252},
    'SignedVolume_raw':  {'category': 'C', 'version': 'raw',  'burn_in': 0},
    'VolGini_raw':       {'category': 'C', 'version': 'raw',  'burn_in': 0},
    'PriceVolCorr_raw':  {'category': 'C', 'version': 'raw',  'burn_in': 0},
    'SignedVolume_z252': {'category': 'C', 'version': 'z252', 'burn_in': 504},
    'VolGini_z252':      {'category': 'C', 'version': 'z252', 'burn_in': 504},
    'PriceVolCorr_z252': {'category': 'C', 'version': 'z252', 'burn_in': 504},
    'BV':                {'category': 'D', 'version': 'main', 'burn_in': 0},
    'Jump':              {'category': 'D', 'version': 'main', 'burn_in': 0},
    'Jump_raw':          {'category': 'D', 'version': 'raw',  'burn_in': 0},
    'SignedJump':        {'category': 'D', 'version': 'main', 'burn_in': 0},
    'VR_5_1':            {'category': 'E', 'version': 'main', 'burn_in': 0},
    'Overnight':         {'category': 'E', 'version': 'main', 'burn_in': 0},
    'Ret_1d':  {'category': 'E', 'version': 'main', 'burn_in': 1},
    'Mom_20d': {'category': 'E', 'version': 'main', 'burn_in': 20},
    'Mom_60d': {'category': 'E', 'version': 'main', 'burn_in': 60},
}

meta_df = pd.DataFrame(factor_meta).T
meta_df.index.name = 'factor'
print(meta_df)

ordered_cols = [c for c in factor_meta.keys() if c in daily.columns]
F = daily[ordered_cols].copy()



F.to_parquet(OUTPUT_DIR / 'spy_factors_step2.parquet')
meta_df.to_csv(OUTPUT_DIR / 'spy_factors_step2_meta.csv')

print(f"\nFactor matrix F shape: {F.shape}")
print(f"Saved to: {OUTPUT_DIR / 'spy_factors_step2.parquet'}")
print(f"Meta saved to: {OUTPUT_DIR / 'spy_factors_step2_meta.csv'}")

                  category version burn_in
factor                                    
RV_open30                A    main       0
RV_close30               A    main       0
RV_close30_nc            A      nc       0
VolPattern               A    main       0
PathEff                  B    main       0
VWAP_Cross               B    main       0
CloseLoc                 B    main       0
SignedVolume             C    main     252
VolGini                  C    main     252
PriceVolCorr             C    main     252
SignedVolume_raw         C     raw       0
VolGini_raw              C     raw       0
PriceVolCorr_raw         C     raw       0
SignedVolume_z252        C    z252     504
VolGini_z252             C    z252     504
PriceVolCorr_z252        C    z252     504
BV                       D    main       0
Jump                     D    main       0
Jump_raw                 D     raw       0
SignedJump               D    main       0
VR_5_1                   E    main       0
Overnight  